# 05 — HuggingFace `evaluate` 評估函式庫（2026 版）

## 學習目標

1. 了解 `evaluate` 函式庫的用途與設計哲學，以及它如何取代各處零散的自製指標計算。
2. 掌握四種使用模式：全局計算、疊代計算、批次疊代計算、多指標組合。
3. 學會用 `compute_metrics` 回呼將 `evaluate` 指標無縫接入 `Trainer`，這是 2026 標準路徑。
4. 理解分類、回歸、文本生成三種任務的指標選擇邏輯，並避免常見誤用（例如把回歸當分類的 threshold trick）。
5. 用 `radar_plot` 做多模型視覺化比較。

## 前置條件

- 完成 `01-Component/04finetune/` 模組（理解微調流程）。
- 本 notebook 產出的 `compute_metrics` 樣板可直接複製進 `Trainer` 的 `training_args` 設定中。

## 與相鄰模組的銜接

- 上一章：[../04finetune/](../04finetune/)（模型微調）
- 下一章：[../../02-NLP-Tasks/](../../02-NLP-Tasks/)（任務實戰，將把本章指標接進完整的訓練迴圈）
- 多模態任務的評測邏輯見 [../../05-Multimodal/](../../05-Multimodal/)（BLEU/METEOR 改用 sacrebleu + bertscore）

## 環境安裝與版本鎖定

`evaluate` 依賴 `datasets` 的型別系統，兩者版本需要配套。
以下同時列出整個 repo 的最小相容版本需求，方便複製進 `requirements.txt`。

In [ ]:
# Version-pinned install — copy this block into requirements.txt for reproducibility
# evaluate>=0.4 brings the modern evaluate.combine() and radar_plot API
# datasets>=3.0 is required for load_dataset batched map and train_test_split stratify
!pip install \
    "evaluate>=0.4" \
    "datasets>=3.0" \
    "transformers>=4.46" \
    "torch>=2.4" \
    "scikit-learn>=1.4" \
    "matplotlib>=3.9" \
    "rouge_score>=0.1.2" \
    --quiet

In [ ]:
import evaluate
import datasets
import torch
import sklearn
import matplotlib

print(f"evaluate  : {evaluate.__version__}")
print(f"datasets  : {datasets.__version__}")
print(f"torch     : {torch.__version__}")
print(f"sklearn   : {sklearn.__version__}")
print(f"matplotlib: {matplotlib.__version__}")

## 1. `evaluate` 函式庫概覽

### 為什麼要用 `evaluate`？

`evaluate` 的設計目標是讓指標計算成為**可組合、可疊代、跨分散式一致**的統一抽象，解決以下三個實際問題：

1. **一致性**：指標的 `average` 策略、正規化方式集中定義在 Hub 版本中，不同 notebook、不同工程師的結果可直接比較。
2. **疊代支援**：分散式訓練或流式評估時，`.add_batch()` 會在背後自動處理 `gather()`，無需手動合併結果。
3. **與 `Trainer` 整合**：`Trainer.compute_metrics` 要求特定介面，`evaluate` 物件可直接包成符合該介面的回呼，無需每次手寫包裝函式。

## 2. 查看支援的評估模組

`evaluate.list_evaluation_modules()` 會列出 HuggingFace Hub 上所有可用的指標、比較器、測量工具。

- `module_type` 可選 `"metric"`（指標）、`"comparison"`（模型比較）、`"measurement"`（模型性質測量，如模型大小）。
- `include_community=True` 會包含社群貢獻的指標（可能不經 HF 官方審查）。

In [ ]:
# List only official HF comparison modules (e.g. McNemar test, bootstrap)
compare_modules = evaluate.list_evaluation_modules(
    module_type="comparison",
    include_community=False,
    with_details=True,
)
for m in compare_modules:
    print(m)

In [ ]:
# List all official metric modules — this is the most commonly used type
metric_modules = evaluate.list_evaluation_modules(
    module_type="metric",
    include_community=False,
    with_details=True,
)
print(f"Total official metrics: {len(metric_modules)}")
for m in metric_modules[:20]:   # show first 20 to avoid scrolling
    print(m)

## 3. 載入評估模組並查看說明文件

`evaluate.load()` 會從 HuggingFace Hub 下載指標定義（首次執行後快取在本機）。
每個指標物件都帶有完整的文件屬性，無需另開瀏覽器查文件。

| 屬性 | 說明 |
|---|---|
| `description` | 指標的簡短說明 |
| `citation` | BibTeX 引用字串 |
| `features` | 輸入的型別定義（`references` / `predictions` 各需要什麼格式）|
| `inputs_description` | 等同 docstring，列出所有參數 |
| `homepage` | 指標的官方頁面 |
| `license` | 授權條款 |

In [ ]:
accuracy = evaluate.load("accuracy")

print("=== description ===")
print(accuracy.description)

print("\n=== inputs_description ===")
print(accuracy.inputs_description)

print("\n=== features ===")
print(accuracy.features)

## 4. 三種計算模式

`evaluate` 支援三種計算模式，對應不同的使用場景：

| 模式 | API | 適用場景 |
|---|---|---|
| 全局計算 | `.compute(predictions=..., references=...)` | 所有預測/標籤已在記憶體中，一次傳入 |
| 疊代計算 | `.add()` + `.compute()` | 每次只處理一筆，記憶體有限或流式場景 |
| 批次疊代計算 | `.add_batch()` + `.compute()` | 每次處理一個 mini-batch，最常見的實際訓練場景 |

### 4.1 全局計算（一次傳入所有結果）

In [ ]:
accuracy = evaluate.load("accuracy")

# Pass all predictions and references at once
results = accuracy.compute(
    references=[0, 1, 2, 0, 1, 2],
    predictions=[0, 1, 1, 2, 1, 0],
)
print(results)   # {'accuracy': 0.5}

### 4.2 疊代計算（逐筆累積）

適合記憶體有限、或在 streaming dataset 上跑評估的情境。
每呼叫一次 `.add()` 只送入一對 `(reference, prediction)`。

In [ ]:
accuracy = evaluate.load("accuracy")

for ref, pred in zip([0, 1, 0, 1], [1, 0, 0, 1]):
    accuracy.add(references=ref, predictions=pred)

result = accuracy.compute()
print(result)   # {'accuracy': 0.5}

### 4.3 批次疊代計算（逐 batch 累積）

這是在 `Trainer` evaluate loop 外自行評估時最常用的模式。
每次呼叫 `.add_batch()` 傳入一個 mini-batch 的 references 和 predictions。

In [ ]:
# batch example: two batches of size 2
# batch 1: refs=[0,1], preds=[1,0] — 0 correct
# batch 2: refs=[0,1], preds=[0,1] — 2 correct
accuracy = evaluate.load("accuracy")

for refs, preds in zip([[0, 1], [0, 1]], [[1, 0], [0, 1]]):
    accuracy.add_batch(references=refs, predictions=preds)

result = accuracy.compute()
print(result)   # {'accuracy': 0.5}  (2 correct out of 4)

## 5. 多指標組合計算（`evaluate.combine`）

分類任務通常需要同時報告 accuracy、precision、recall、F1。
`evaluate.combine()` 把多個指標包成一個物件，一次 `.compute()` 得到所有指標。

```python
clf_metrics = evaluate.combine(["accuracy", "f1", "recall", "precision"])

def compute_metrics(pred):
    preds = pred.predictions.argmax(-1)
    return clf_metrics.compute(predictions=preds, references=pred.label_ids)
```

優勢：指標定義集中在一處、`average` 策略由 Hub 版本決定、未來加指標只需改一行。

In [ ]:
# Combine multiple classification metrics into a single object
clf_metrics = evaluate.combine(["accuracy", "f1", "recall", "precision"])
print(type(clf_metrics))
print(dir(clf_metrics))

In [ ]:
# Compute all metrics at once — binary classification example
results = clf_metrics.compute(
    predictions=[0, 1, 0],
    references=[0, 1, 1],
)
print(results)

### 5.1 接入 `Trainer.compute_metrics`（2026 標準路徑）

`Trainer` 在每個 eval step 結束後會呼叫 `compute_metrics(EvalPrediction)` 回呼。
`EvalPrediction` 帶有兩個欄位：
- `pred.predictions`：模型輸出的 logits，shape `(n_samples, n_classes)`
- `pred.label_ids`：真實標籤，shape `(n_samples,)`

下面是可直接複製進微調 notebook 的標準樣板：

In [ ]:
import numpy as np
import evaluate

# --- 2026 standard compute_metrics template for classification ---

# Load once at module level to avoid repeated Hub requests
_clf_metrics = evaluate.combine(["accuracy", "f1", "recall", "precision"])

def compute_metrics_classification(eval_pred):
    """Standard compute_metrics for multi-class classification.

    Compatible with Trainer / SFTTrainer via TrainingArguments(compute_metrics=...).
    """
    logits, labels = eval_pred
    # argmax over class dimension to get predicted class indices
    predictions = np.argmax(logits, axis=-1)
    return _clf_metrics.compute(
        predictions=predictions,
        references=labels,
        # average='macro' treats all classes equally regardless of support
        average="macro",
    )

# --- Sanity check ---
from transformers import EvalPrediction

fake_logits = np.array([[2.0, 0.1], [0.3, 1.8], [1.5, 0.2]])
fake_labels = np.array([0, 1, 1])
result = compute_metrics_classification(EvalPrediction(predictions=fake_logits, label_ids=fake_labels))
print(result)

### 5.2 混淆矩陣補充分析

單一數值指標（accuracy / F1）只能說「整體好不好」，無法定位「哪一類別出問題」。
推薦在 evaluate 計算後補一張混淆矩陣做錯誤分析。

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
import matplotlib.pyplot as plt

def plot_confusion_matrix(y_true, y_pred, class_names=None):
    """Plot normalized confusion matrix for error analysis."""
    cm = confusion_matrix(y_true, y_pred, normalize="true")
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    fig, ax = plt.subplots(figsize=(6, 5))
    disp.plot(ax=ax, colorbar=True, cmap="Blues")
    ax.set_title("Normalized Confusion Matrix")
    plt.tight_layout()
    plt.show()

# Example: 3-class sentiment (negative / neutral / positive)
y_true = [0, 1, 2, 0, 1, 2, 0, 2, 1]
y_pred = [0, 1, 1, 2, 1, 0, 0, 2, 1]
plot_confusion_matrix(y_true, y_pred, class_names=["negative", "neutral", "positive"])

## 6. 回歸任務的正確指標選擇

回歸模型的輸出是連續數值（例如語句相似度分數 0.0～1.0），應使用連續性指標評估：

- **MSE（均方誤差）**：衡量預測值與真實值的平均平方偏差，對大偏差更敏感。
- **Pearson r**：衡量預測值與真實值的線性相關程度，不受數值量級影響，適合排名一致性分析。

對回歸模型使用 accuracy（需先套 threshold 轉分類）是指標語意錯誤：這樣做會丟失連續性資訊，且 threshold 的選定是任意的，結果不具可比性。

In [ ]:
# Correct metrics for regression tasks (e.g. semantic similarity scoring)
mse_metric = evaluate.load("mse")
pearsonr_metric = evaluate.load("pearsonr")

# Example: sentence similarity scores (range 0.0 ~ 1.0)
y_true  = [0.8, 0.3, 0.9, 0.1, 0.6]
y_pred  = [0.75, 0.35, 0.85, 0.15, 0.55]

mse_result     = mse_metric.compute(predictions=y_pred, references=y_true)
pearson_result = pearsonr_metric.compute(predictions=y_pred, references=y_true)

print(f"MSE     : {mse_result}")
print(f"Pearson r: {pearson_result}")

# --- 2026 standard compute_metrics template for regression ---
_mse     = evaluate.load("mse")
_pearson = evaluate.load("pearsonr")

def compute_metrics_regression(eval_pred):
    """Standard compute_metrics for regression tasks.

    Model output: single logit per sample (no argmax needed).
    """
    logits, labels = eval_pred
    # squeeze the last dim if shape is (n, 1)
    predictions = logits.squeeze(-1) if logits.ndim == 2 else logits
    mse = _mse.compute(predictions=predictions, references=labels)["mse"]
    pcc = _pearson.compute(predictions=predictions, references=labels)["pearsonr"]
    return {"mse": mse, "pearsonr": pcc}

## 7. 文本生成任務：ROUGE（摘要評估）

ROUGE（Recall-Oriented Understudy for Gisting Evaluation）是衡量摘要品質最常用的指標系列：

| 指標 | 說明 |
|---|---|
| `rouge1` | 1-gram 重疊率（詞彙覆蓋） |
| `rouge2` | 2-gram 重疊率（相鄰詞對的覆蓋） |
| `rougeL` | 最長公共子序列（保持詞序） |
| `rougeLsum` | 句子層級的 rougeL（摘要任務專用，考慮分句符號）|

使用 `evaluate.load("rouge")` 載入官方實作，底層依賴 `rouge_score` 套件，同時支援英文與中文（中文需先將文字切分為字元序列）。

In [ ]:
# evaluate.load('rouge') uses the rouge_score package internally
# works for both English and CJK text (CJK needs pre-tokenization into chars)
rouge = evaluate.load("rouge")

predictions = [
    "The cat sat on the mat.",
    "The quick brown fox jumped over the lazy dog.",
]
references = [
    "The cat is sitting on the mat.",
    "A fast brown fox leaped over a sleeping dog.",
]

results = rouge.compute(predictions=predictions, references=references)
for k, v in results.items():
    print(f"{k}: {v:.4f}")

In [ ]:
# For Chinese text: tokenize into characters first
# (rouge_score treats space-separated tokens as n-gram units)
def tokenize_chinese(text: str) -> str:
    """Insert spaces between every character for CJK ROUGE scoring."""
    return " ".join(list(text))

chinese_preds = [tokenize_chinese("今天天氣很好，適合出門散步。")]
chinese_refs  = [tokenize_chinese("今天的天氣非常好，很適合外出走走。")]

chinese_rouge = rouge.compute(predictions=chinese_preds, references=chinese_refs)
for k, v in chinese_rouge.items():
    print(f"{k}: {v:.4f}")

## 8. 多模型結果視覺化（Radar Plot）

`evaluate.visualization.radar_plot` 提供雷達圖，適合在同一個維度上比較多個模型。

注意事項：
- 雷達圖假設「所有軸的數值越大越好」，若有「越小越好」的指標（如 latency、MSE），需事先做倒數或標準化轉換。
- 軸的數值範圍影響視覺解讀，指標之間量級差異大時慎用。

In [ ]:
from evaluate.visualization import radar_plot
import matplotlib.pyplot as plt

# Example: comparing 4 model checkpoints on 4 metrics
# Note: latency_in_seconds is intentionally excluded here because
# radar_plot treats all axes as "higher is better".
# If you want to include latency, convert it: latency_score = 1 / latency_seconds
data = [
    {"accuracy": 0.99, "precision": 0.80, "f1": 0.95, "recall": 0.91},
    {"accuracy": 0.98, "precision": 0.87, "f1": 0.91, "recall": 0.88},
    {"accuracy": 0.98, "precision": 0.78, "f1": 0.88, "recall": 0.85},
    {"accuracy": 0.88, "precision": 0.78, "f1": 0.81, "recall": 0.79},
]
model_names = ["Model A (large)", "Model B (base)", "Model C (distilled)", "Model D (tiny)"]

plot = radar_plot(data=data, model_names=model_names)
plt.title("Model Comparison — Classification Metrics")
plt.tight_layout()
plt.show()

## 9. 完整 `compute_metrics` 樣板索引

本節彙整三類任務的標準樣板，可直接複製進 `Trainer` 呼叫。

In [ ]:
import numpy as np
import evaluate

# =====================================================================
# Template A: Multi-class classification
# =====================================================================
_clf = evaluate.combine(["accuracy", "f1", "recall", "precision"])

def compute_metrics_clf(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return _clf.compute(predictions=preds, references=labels, average="macro")


# =====================================================================
# Template B: Binary classification
# =====================================================================
_bin_clf = evaluate.combine(["accuracy", "f1", "recall", "precision"])

def compute_metrics_binary(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)   # or (logits[:, 1] > 0.5).astype(int)
    return _bin_clf.compute(predictions=preds, references=labels, average="binary")


# =====================================================================
# Template C: Regression (e.g. semantic similarity, STS-B)
# =====================================================================
_mse     = evaluate.load("mse")
_pearson = evaluate.load("pearsonr")

def compute_metrics_reg(eval_pred):
    logits, labels = eval_pred
    preds = logits.squeeze(-1) if logits.ndim == 2 else logits
    return {
        "mse":      _mse.compute(predictions=preds, references=labels)["mse"],
        "pearsonr": _pearson.compute(predictions=preds, references=labels)["pearsonr"],
    }


# =====================================================================
# Template D: Text generation / summarization (ROUGE)
# =====================================================================
# Note: for seq2seq models use tokenizer.batch_decode() before computing.
_rouge = evaluate.load("rouge")

def compute_metrics_rouge(eval_pred, tokenizer):
    """For use with Seq2SeqTrainer (requires predict_with_generate=True)."""
    preds, labels = eval_pred
    # replace -100 padding token ids with pad_token_id for decoding
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_preds  = tokenizer.batch_decode(preds,  skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    return _rouge.compute(predictions=decoded_preds, references=decoded_labels)


print("All compute_metrics templates defined.")

## 小結

| 主題 | 2026 做法 |
|---|---|
| 分類指標計算 | `evaluate.combine([...])` 統一定義，集中 `average` 策略 |
| 回歸指標 | `evaluate.load("mse")` + `evaluate.load("pearsonr")` |
| 中文摘要 ROUGE | `evaluate.load("rouge")` + 字元切分前處理 |
| 接入 Trainer | 複製本章 `compute_metrics` 樣板，傳入 `TrainingArguments` |
| 多模型比較 | `radar_plot` + 統一 data 格式 |

## 練習題

1. 用 `evaluate.combine(["accuracy", "f1"])` 計算一個五分類問題（classes 0~4）的 macro F1，並畫出混淆矩陣。說明哪個類別最容易被混淆。

2. 以 `evaluate.load("pearsonr")` 評估一個語句相似度模型（輸出為 0~5 的連續分數）。為什麼 Pearson r 比 accuracy 更適合這個任務？

3. 試著在上方 `compute_metrics_rouge` 中加入 `rougeLsum` 指標，並解釋 `rougeL` 和 `rougeLsum` 在摘要任務上的差異。

4. 把 `latency_in_seconds` 轉換成 `throughput_score = 1 / latency` 後，再跟 accuracy / precision / F1 一起放進 `radar_plot`，比較轉換前後的視覺差異。

5. 閱讀 `evaluate.load("bertscore")` 的 `inputs_description`，說明 BERTScore 相對於 ROUGE 的優缺點，並列舉一個 ROUGE 會誤判但 BERTScore 較準確的例子。